# Cosine Similarity Calculated Mathematically in Python

**Environment:** Google Colab / Local Jupyter Notebook (Python 3.9+)  
**Domain:** Vector Geometry, Distance Metrics & High-Dimensional Feature Spaces

---

### What You Will Learn
1. **Mathematical Foundations:** The Dot Product, $L_2$ Euclidean Norm, and the Cauchy-Schwarz inequality.
2. **From-Scratch Python Implementation:** Writing safe, zero-dependency manual functions for dot product, norm, and cosine similarity.
3. **Geometric Intuition:** Visualizing vectors, unit-circle projections, and why cosine similarity measures **orientation, not magnitude**.
4. **Vectorized NumPy & Batch Matrix Operations:** Implementing $S = \frac{X Y^T}{\|X\| \|Y\|}$ for high-throughput batch scoring.
5. **Metric Comparison:** Cosine vs. Euclidean ($L_2$) vs. Manhattan ($L_1$) vs. Dot Product.
6. **Real-World Application:** Building a Candidate-Skill Matching Engine.
7. **Advanced Practice Exercises:** Angular Distance, Centered Cosine Similarity (Pearson correlation), and custom batch functions.

In [ ]:
# Install required dependencies
!pip install -q numpy scikit-learn matplotlib seaborn pandas

## 1. The Mathematical Formulation

Let $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ be two non-zero vectors in an $n$-dimensional Euclidean space.

### 1.1 The Dot Product (Scalar Product)
The dot product represents the sum of the element-wise products:
$$
\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^n u_i v_i = u_1 v_1 + u_2 v_2 + \dots + u_n v_n
$$

Geometrically, the dot product satisfies:
$$
\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\|_2 \|\mathbf{v}\|_2 \cos(\theta)
$$
where $\theta$ is the angle between the two vectors.

### 1.2 Euclidean Norm (Magnitude)
The length (or $L_2$ norm) of vector $\mathbf{u}$ is:
$$
\|\mathbf{u}\|_2 = \sqrt{\sum_{i=1}^n u_i^2} = \sqrt{\mathbf{u} \cdot \mathbf{u}}
$$

### 1.3 Cosine Similarity
Rearranging the geometric formula gives the **Cosine Similarity**:
$$
S_C(\mathbf{u}, \mathbf{v}) = \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \frac{\sum_{i=1}^n u_i v_i}{\sqrt{\sum_{i=1}^n u_i^2} \sqrt{\sum_{i=1}^n v_i^2}}
$$
- $S_C = +1 \implies$ Vectors point in the exact same direction ($\theta = 0^\circ$).
- $S_C = 0 \implies$ Vectors are orthogonal ($\theta = 90^\circ$, completely unrelated).
- $S_C = -1 \implies$ Vectors point in diametrically opposite directions ($\theta = 180^\circ$).

### 1.4 Cosine Distance
Vector databases frequently convert similarity into distance:
$$
D_C(\mathbf{u}, \mathbf{v}) = 1 - S_C(\mathbf{u}, \mathbf{v})
$$

In [ ]:
# Define sample 5-dimensional candidate skill vectors
# Dimensions: [AI_ML, Cloud_Infra, Data_Eng, DevOps, Cybersecurity]
vector_u = [5, 4, 2, 1, 0]   # Heavy AI & Cloud engineer
vector_v = [4, 3, 3, 2, 1]   # Balanced Full-Stack Data/AI engineer

print("Vector u (Skill Profile 1):", vector_u)
print("Vector v (Skill Profile 2):", vector_v)

## 2. Pure Python Implementation (Zero Dependencies)

We now implement the dot product, Euclidean norm, and cosine similarity using pure Python loops and standard `math` functions.

In [ ]:
import math

def manual_dot_product(u, v):
    """Calculates dot product with dimension validation."""
    if len(u) != len(v):
        raise ValueError(f"Vectors must have identical dimensions! Got {len(u)} and {len(v)}")
    
    dot = 0.0
    for a, b in zip(u, v):
        dot += a * b
    return dot

def manual_magnitude(u):
    """Calculates Euclidean L2 norm of a vector."""
    squared_sum = sum(x ** 2 for x in u)
    return math.sqrt(squared_sum)

def manual_cosine_similarity(u, v):
    """Calculates Cosine Similarity, handling zero-division safely."""
    mag_u = manual_magnitude(u)
    mag_v = manual_magnitude(v)
    
    if mag_u == 0.0 or mag_v == 0.0:
        raise ValueError("Cannot calculate cosine similarity for zero-magnitude vector!")
        
    dot = manual_dot_product(u, v)
    similarity = dot / (mag_u * mag_v)
    
    # Clip to avoid float precision anomalies beyond [-1.0, 1.0]
    similarity = max(-1.0, min(1.0, similarity))
    angle_rad = math.acos(similarity)
    angle_deg = math.degrees(angle_rad)
    
    return similarity, angle_deg

# Execute calculation
dot_val = manual_dot_product(vector_u, vector_v)
mag_u_val = manual_magnitude(vector_u)
mag_v_val = manual_magnitude(vector_v)
sim_val, angle_val = manual_cosine_similarity(vector_u, vector_v)

print(f"Dot Product (u . v)    : {dot_val:.4f}")
print(f"Magnitude ||u||        : {mag_u_val:.4f}")
print(f"Magnitude ||v||        : {mag_v_val:.4f}")
print(f"Cosine Similarity      : {sim_val:.6f}")
print(f"Cosine Distance (1 - S): {1 - sim_val:.6f}")
print(f"Separation Angle (deg) : {angle_val:.2f}°")

## 3. Verification with NumPy and Scikit-Learn

In production environments, pure Python loops are replaced by optimized SIMD/BLAS vectorization provided by NumPy and Scikit-Learn.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cos_sim

u_np = np.array(vector_u, dtype=float)
v_np = np.array(vector_v, dtype=float)

# NumPy vectorized calculation
numpy_dot = np.dot(u_np, v_np)
numpy_mag_u = np.linalg.norm(u_np)
numpy_mag_v = np.linalg.norm(v_np)
numpy_sim = numpy_dot / (numpy_mag_u * numpy_mag_v)

# Scikit-learn expects 2D matrices (n_samples, n_features)
sklearn_sim = sklearn_cos_sim(u_np.reshape(1, -1), v_np.reshape(1, -1))[0, 0]

print(f"Manual Python Result : {sim_val:.8f}")
print(f"NumPy Vectorized     : {numpy_sim:.8f}")
print(f"Scikit-Learn Result  : {sklearn_sim:.8f}")
assert math.isclose(sim_val, numpy_sim, rel_tol=1e-7), "Verification failed!"
print("\nAll implementations match within floating-point tolerance!")

### 3.1 Batch Pairwise Cosine Similarity Function
Vector databases calculate similarities between a query matrix $X \in \mathbb{R}^{M 	imes D}$ and a document corpus $Y \in \mathbb{R}^{N 	imes D}$.
$$
S = rac{X Y^T}{\|X\| \|Y\|}
$$

In [ ]:
def batch_cosine_similarity(X, Y):
    """
    Vectorized pairwise cosine similarity between matrix X (M x D) and Y (N x D).
    Returns similarity matrix of shape (M x N).
    """
    X = np.asarray(X)
    Y = np.asarray(Y)
    
    # Compute norms along feature axis
    norm_X = np.linalg.norm(X, axis=1, keepdims=True)
    norm_Y = np.linalg.norm(Y, axis=1, keepdims=True)
    
    # Avoid division by zero
    norm_X = np.where(norm_X == 0, 1e-12, norm_X)
    norm_Y = np.where(norm_Y == 0, 1e-12, norm_Y)
    
    # Dot product between all pairs
    dot_products = np.dot(X, Y.T)
    return dot_products / np.dot(norm_X, norm_Y.T)

# Test batch computation
matrix_A = np.array([[5, 4, 2, 1, 0], [1, 0, 0, 5, 4]])
matrix_B = np.array([[4, 3, 3, 2, 1], [0, 0, 1, 4, 5], [5, 4, 2, 1, 0]])

batch_res = batch_cosine_similarity(matrix_A, matrix_B)
print("Batch Cosine Similarity Matrix (2 x 3):")
print(np.round(batch_res, 4))

### 3.2 Micro-Benchmark: Pure Python vs. NumPy
Let us benchmark the performance difference over 5,000 vector pairs.

In [ ]:
import time

# Generate 5,000 synthetic 64-dimensional vectors
dim = 64
n_vectors = 5000
test_data_u = np.random.rand(n_vectors, dim)
test_data_v = np.random.rand(n_vectors, dim)

# Benchmark Pure Python
list_u = test_data_u.tolist()
list_v = test_data_v.tolist()

t0 = time.perf_counter()
for i in range(n_vectors):
    _ = manual_cosine_similarity(list_u[i], list_v[i])
t_py = time.perf_counter() - t0

# Benchmark NumPy
t0 = time.perf_counter()
dots = np.sum(test_data_u * test_data_v, axis=1)
norms = np.linalg.norm(test_data_u, axis=1) * np.linalg.norm(test_data_v, axis=1)
_ = dots / norms
t_np = time.perf_counter() - t0

print(f"Pure Python Time (5,000 pairs) : {t_py:.4f} seconds")
print(f"Vectorized NumPy Time           : {t_np:.4f} seconds")
print(f"NumPy Speedup Factor            : {t_py / t_np:.1f}x faster!")

## 4. Geometric Visualization: Why Scale Invariance Matters

A pivotal feature of Cosine Similarity is **Scale Invariance**:
$$
S_C(k \cdot \mathbf{u}, \mathbf{v}) = S_C(\mathbf{u}, \mathbf{v}) \quad (orall k > 0)
$$

Scaling a vector alters its Euclidean magnitude, but leaves its spatial direction unchanged. In NLP, this means a short 200-word article and a 2,000-word deep-dive on the same topic will have high cosine similarity!

In [ ]:
import matplotlib.pyplot as plt

# 2D Vectors to demonstrate geometric intuition
v1 = np.array([2, 3])
v2 = np.array([3, 1])
v1_scaled = v1 * 3.5  # Same direction, 3.5x longer

plt.figure(figsize=(9, 7))

# Plot coordinate axes
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)

# Plot unit circle for reference
circle = plt.Circle((0, 0), 1, color='lightblue', fill=False, linestyle=':', label='Unit Circle (L2 = 1)')
plt.gca().add_patch(circle)

# Plot vectors as arrows from origin
plt.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='#2563eb', label='v1 [2, 3]')
plt.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='#dc2626', label='v2 [3, 1]')
plt.quiver(0, 0, v1_scaled[0], v1_scaled[1], angles='xy', scale_units='xy', scale=1, color='#9333ea', linestyle='dashed', alpha=0.6, label='v1_scaled [7, 10.5] (3.5x)')

plt.xlim(-1, 9)
plt.ylim(-1, 12)
plt.gca().set_aspect('equal', adjustable='box')
plt.title("Vector Orientation and Scale Invariance", fontsize=13, weight="bold")
plt.xlabel("X1")
plt.ylabel("X2")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

# Verification metrics
sim_v1_v2 = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
sim_scaled_v2 = np.dot(v1_scaled, v2) / (np.linalg.norm(v1_scaled) * np.linalg.norm(v2))
dist_euclid_original = np.linalg.norm(v1 - v2)
dist_euclid_scaled = np.linalg.norm(v1_scaled - v2)

print(f"Cosine Sim (v1, v2)        : {sim_v1_v2:.4f}")
print(f"Cosine Sim (v1_scaled, v2) : {sim_scaled_v2:.4f}  <-- EXACTLY IDENTICAL!")
print(f"Euclidean Dist (v1, v2)    : {dist_euclid_original:.4f}")
print(f"Euclidean Dist (scaled, v2): {dist_euclid_scaled:.4f}  <-- EXPLODES!")

## 5. Metric Comparison: Cosine vs. Euclidean vs. Manhattan vs. Dot Product

| Metric | Formula | Value Range | Properties |
|---|---|---|---|
| **Cosine Similarity** | $\frac{u \cdot v}{\|u\| \|v\|}$ | $[-1, +1]$ | Measures angle; ignores magnitude |
| **Cosine Distance** | $1 - S_C(u, v)$ | $[0, 2]$ | Lower is closer; standard in ChromaDB |
| **Euclidean Distance ($L_2$)** | $\sqrt{\sum (u_i - v_i)^2}$ | $[0, \infty)$ | Measures straight-line geometric distance |
| **Manhattan Distance ($L_1$)** | $\sum |u_i - v_i|$ | $[0, \infty)$ | Grid-based taxi distance |
| **Dot Product** | $\sum u_i v_i$ | $(-\infty, \infty)$ | Combines angle and magnitudes |

In [ ]:
def compare_metrics(u, v):
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)
    
    cos_sim = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    cos_dist = 1.0 - cos_sim
    l2_dist = np.linalg.norm(u - v)
    l1_dist = np.sum(np.abs(u - v))
    dot_prod = np.dot(u, v)
    
    return {
        "Cosine Sim": round(float(cos_sim), 4),
        "Cosine Dist": round(float(cos_dist), 4),
        "Euclidean (L2)": round(float(l2_dist), 4),
        "Manhattan (L1)": round(float(l1_dist), 4),
        "Dot Product": round(float(dot_prod), 4)
    }

cases = {
    "Identical vectors": ([2, 4, 6], [2, 4, 6]),
    "Scaled vectors (same direction)": ([1, 2, 3], [5, 10, 15]),
    "Orthogonal vectors (independent)": ([1, 0, 0], [0, 1, 0]),
    "Opposite vectors": ([3, 2, 1], [-3, -2, -1]),
    "Shifted components": ([5, 1, 0], [1, 5, 0])
}

rows = []
for label, (u, v) in cases.items():
    metrics = compare_metrics(u, v)
    metrics["Scenario"] = label
    rows.append(metrics)

df_cases = pd.DataFrame(rows).set_index("Scenario")
display(df_cases)

## 6. Real-World Application: Candidate Skill Matching Engine

Let us build a candidate matcher. We have a target job role profile and a database of candidate skill scores across 6 technical categories:
`[Deep Learning, Cloud Architecture, Python, DevOps, SQL/Databases, System Design]`

In [ ]:
categories = ["DL", "Cloud", "Python", "DevOps", "SQL", "SysDesign"]

# Required job profile
target_job = np.array([5, 4, 5, 3, 3, 4], dtype=float)

candidates = {
    "Alice (Senior ML Engineer)":   [5, 4, 5, 3, 4, 4],
    "Bob (Junior Python Dev)":      [1, 1, 4, 1, 2, 1],
    "Charlie (DevOps Specialist)":  [1, 5, 3, 5, 3, 3],
    "Diana (Data Architect)":       [3, 5, 4, 2, 5, 5],
    "Evan (ML Researcher)":         [5, 2, 4, 1, 2, 3]
}

match_results = []
for name, scores in candidates.items():
    cand_vec = np.array(scores, dtype=float)
    cos_sim = np.dot(target_job, cand_vec) / (np.linalg.norm(target_job) * np.linalg.norm(cand_vec))
    l2_dist = np.linalg.norm(target_job - cand_vec)
    
    match_results.append({
        "Candidate": name,
        "Cosine Match Score": round(float(cos_sim) * 100, 2),  # Percentage match
        "Euclidean Gap": round(float(l2_dist), 2)
    })

df_matches = pd.DataFrame(match_results).sort_values("Cosine Match Score", ascending=False)
display(df_matches.reset_index(drop=True))

## 7. Practice Exercises (Solved)

### Exercise 1: Angular Distance Function
Cosine distance does not satisfy the triangle inequality. **Angular Distance** does:
$$
D_	heta(\mathbf{u}, \mathbf{v}) = rac{rccos(S_C(\mathbf{u}, \mathbf{v}))}{\pi} \in [0, 1]
$$

In [ ]:
def angular_distance(u, v):
    """Calculates normalized Angular Distance in [0, 1]."""
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)
    sim = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    sim = np.clip(sim, -1.0, 1.0)
    return np.arccos(sim) / np.pi

print("Angular Distance (Identical) :", angular_distance([2, 3], [2, 3]))
print("Angular Distance (Orthogonal):", angular_distance([1, 0], [0, 1]))
print("Angular Distance (Opposite)  :", angular_distance([2, 3], [-2, -3]))

### Exercise 2: Centered Cosine Similarity (Pearson Correlation)
When features have different mean baselines, centering them by subtracting the mean produces the **Pearson Correlation Coefficient**:
$$
S_{	ext{centered}}(u, v) = rac{(u - ar{u}) \cdot (v - ar{v})}{\|u - ar{u}\|_2 \|v - ar{v}\|_2}
$$

In [ ]:
def centered_cosine_similarity(u, v):
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)
    u_centered = u - np.mean(u)
    v_centered = v - np.mean(v)
    return np.dot(u_centered, v_centered) / (np.linalg.norm(u_centered) * np.linalg.norm(v_centered))

cand_1 = [5, 4, 5, 3, 4, 4]
cand_2 = [3, 2, 3, 1, 2, 2]  # Identical rating shape, but strictly 2 points lower baseline

print("Standard Cosine Sim :", round(float(np.dot(cand_1, cand_2) / (np.linalg.norm(cand_1) * np.linalg.norm(cand_2))), 4))
print("Centered Cosine Sim :", round(float(centered_cosine_similarity(cand_1, cand_2)), 4))

## Key Takeaways

1. **Orientation Over Magnitude:** Cosine similarity measures directional alignment, ignoring vector length.
2. **Normalized Dot Product:** If two vectors are $L_2$-normalized to unit length ($\|u\|_2 = 1$), their dot product equals their cosine similarity ($u \cdot v = \cos(	heta)$). This is why vector search engines pre-normalize embeddings.
3. **Range:** Values range from $-1.0$ (opposite) to $+1.0$ (identical). Text embeddings with non-negative activations fall within $[0, 1]$.
4. **Vector DB Distance:** ChromaDB uses $1 - 	ext{cosine\_similarity}$ as its cosine distance metric.